# Elastic Net regression

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
Elastic Net combines Ridge and Lasso:

$$J=\frac{1}{2n}\sum(y_i-\hat y_i)^2+\alpha\Big(\rho\sum\vert w_j\vert+\frac{1-\rho}{2}\sum w_j^2\Big)$$

- $\rho$ is `l1_ratio`: 1 means pure Lasso, 0 means pure Ridge.
- From Lasso: zero coefficients and feature selection.
- From Ridge: stability with correlated features.

## The problem it solves
Suppose you have three nearly identical features (say temperature in Celsius, Fahrenheit and Kelvin). Lasso keeps one at random and zeroes the other two, and a small change in the data may make it choose another. Elastic Net keeps all three **together** and shares the weight among them (the **grouping effect**).



In [ ]:
import numpy as np
from sklearn.linear_model import Lasso, ElasticNet

rng = np.random.default_rng(0)
t = rng.normal(size=200)
# three almost identical copies of the same signal, plus one useless feature
X = np.c_[t + rng.normal(0, 0.01, 200), t + rng.normal(0, 0.01, 200), t + rng.normal(0, 0.01, 200), rng.normal(size=200)]
y = 3 * t + rng.normal(0, 0.5, 200)

print("Lasso      :", Lasso(alpha=0.1).fit(X, y).coef_.round(2))
print("ElasticNet :", ElasticNet(alpha=0.1, l1_ratio=0.3).fit(X, y).coef_.round(2))



> [Elastic Net coefficient paths with l1_ratio = 0.5.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/enet_path.png)

*Elastic Net coefficient paths with l1_ratio = 0.5.*

[[lab:reg_enet]]

In the lab, move `l1_ratio` from 0 to 1: more coefficients become zero.

## The three methods side by side
| | Ridge | Lasso | Elastic Net |
|---|---|---|---|
| Penalty | $\sum w^2$ | $\sum\vert w\vert$ | Both |
| Zero coefficients | No | Yes | Yes |
| Correlated features | Shares the weight | Picks one | Shares the weight |
| Hyperparameters | α | α | α and l1_ratio |
| Closed formula | Yes | No | No |

## Tuning two hyperparameters
`ElasticNetCV` searches both with cross-validation:



In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_diabetes(return_X_y=True)
m = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], cv=5, random_state=0)).fit(X, y)
print("alpha =", round(m[-1].alpha_, 4), " l1_ratio =", m[-1].l1_ratio_)
print("Non-zero coefficients:", (m[-1].coef_ != 0).sum(), "of", len(m[-1].coef_))



> In practice, if you are unsure which method to use, Elastic Net with `l1_ratio` in [0.1, 0.5, 0.9, 1] is a safe choice.

### Exercise

In the first code, set l1_ratio to 1, 0.5 and 0.1 and compare the first three coefficients.

In [ ]:
# Your answer here